In [ ]:
from micrograd.nn import MLP

In [ ]:
from graphviz import Digraph


def trace(root):
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root, format="svg", rankdir="LR"):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ["LR", "TB"]
    nodes, edges = trace(root)
    dot = Digraph(
        format=format, graph_attr={"rankdir": rankdir}
    )  # , node_attr={'rankdir': 'TB'})

    for n in nodes:
        dot.node(
            name=str(id(n)),
            label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad),
            shape="record",
        )
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [ ]:
# dataset
xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]

# target - desired targets
ys = [1.0, -1.0, -1.0, 1.0]

In [ ]:
# Neural network initialization
nn = MLP(3, [4, 4, 1])
len(nn.parameters())

In [ ]:
rounds = 1000
for k in range(rounds):
    # forward pass
    ypred = [nn(x) for x in xs]

    # loss calculation
    loss = sum([(yout - ygt) ** 2 for ygt, yout in zip(ys, ypred)])

    # zero out gradients
    for p in nn.parameters():
        p.zero_grad()

    # backward pass
    loss.backward()

    # learning
    for p in nn.parameters():
        p.data += -0.1 * p.grad

    if k % 100 == 0:
        print(loss)
    
ys, ypred